# 19A2C — Prepare Deduplicated Cycle-24 AIA Cache Staging

## Purpose

Prepare the exact file lists and local-path mapping needed to stage the frozen Cycle-24 temporal AIA development data onto a GPU VM.

This notebook does **not** download image payloads and does **not** train a model.

It consumes the 19A2B unique-object inventory and:

- verifies all 70,217 source objects are represented once;
- checks that destination basenames are collision-free within each year;
- creates one URI list per year for parallel `gsutil -m cp -I` staging;
- creates the local target-to-frame map used by the training loader;
- writes a reproducible shell script for the GPU VM;
- keeps Cycle-25 completely absent.


In [ ]:
from pathlib import Path
import json, re

import pandas as pd

HOME = Path.home()
PLAN = HOME / "aia19_staging_plan_fast"
OUT = HOME / "aia19_cycle24_cache_prep"
OUT.mkdir(parents=True, exist_ok=True)
LISTS = OUT / "uri_lists"
LISTS.mkdir(exist_ok=True)

OBJECTS = PLAN / "cycle24_unique_aia_objects.csv.gz"
TARGETS = PLAN / "cycle24_temporal_target_to_object_map.csv.gz"

CACHE_ROOT = Path("/mnt/disks/aia-cache/cycle24")

for p in [OBJECTS, TARGETS]:
    if not p.exists():
        raise FileNotFoundError(p)

print("OBJECTS:", OBJECTS)
print("TARGETS:", TARGETS)
print("CACHE_ROOT:", CACHE_ROOT)


## 1. Load and verify the deduplicated object inventory

In [ ]:
objects = pd.read_csv(OBJECTS)
targets = pd.read_csv(TARGETS)

assert len(objects) == 70217, len(objects)
assert len(targets) == 64725, len(targets)
assert int(targets["label_48h_final"].sum()) == 2094

if objects["object_uri"].duplicated().any():
    raise RuntimeError("Duplicate object URI in supposedly unique inventory.")

def year_from_uri(uri: str) -> int:
    m = re.search(r"/samples_npz/(?:[^/]+/)?(20\d{2})/", uri)
    if not m:
        raise ValueError(f"Could not recover year from URI: {uri}")
    return int(m.group(1))

def basename(uri: str) -> str:
    return uri.rstrip("/").split("/")[-1]

objects["source_year"] = objects["object_uri"].map(year_from_uri)
objects["basename"] = objects["object_uri"].map(basename)
objects["local_path"] = objects.apply(
    lambda r: str(CACHE_ROOT / str(int(r["source_year"])) / r["basename"]),
    axis=1,
)

print("Years:", sorted(objects["source_year"].unique().tolist()))
print(objects.groupby("source_year").size().to_string())


## 2. Check local-path collisions

In [ ]:
dups = objects[objects["local_path"].duplicated(keep=False)].sort_values("local_path")
print("Local-path collisions:", len(dups))

if len(dups):
    print(dups[["object_uri", "local_path"]].head(50).to_string(index=False))
    raise RuntimeError("Destination path collision detected; do not stage.")

print("Collision check passed.")


## 3. Write per-year URI lists

In [ ]:
year_counts = {}

for year, group in objects.sort_values("object_uri").groupby("source_year"):
    p = LISTS / f"cycle24_{int(year)}_uris.txt"
    p.write_text("\n".join(group["object_uri"].tolist()) + "\n")
    year_counts[str(int(year))] = int(len(group))
    print(year, len(group), p)

assert sum(year_counts.values()) == 70217


## 4. Convert target URI references to deterministic local cache paths

In [ ]:
uri_to_local = dict(zip(objects["object_uri"], objects["local_path"]))

uri_cols = [
    "history_uri_tminus288",
    "history_uri_tminus192",
    "history_uri_tminus96",
]
local_cols = [
    "local_tminus288",
    "local_tminus192",
    "local_tminus96",
]

local_targets = targets.copy()

for ucol, lcol in zip(uri_cols, local_cols):
    local_targets[lcol] = local_targets[ucol].map(uri_to_local)
    if local_targets[lcol].isna().any():
        raise RuntimeError(f"Missing local-path mapping for {ucol}")

local_targets.to_csv(
    OUT / "cycle24_temporal_targets_local_paths.csv.gz",
    index=False,
    compression="gzip",
)

objects.to_csv(
    OUT / "cycle24_unique_aia_objects_with_local_paths.csv.gz",
    index=False,
    compression="gzip",
)

print("Target rows:", len(local_targets))
print("Object rows:", len(objects))


## 5. Write the GPU-VM staging script

In [ ]:
script = r'''#!/usr/bin/env bash
set -euo pipefail

CACHE_ROOT="/mnt/disks/aia-cache/cycle24"
PREP_DIR="${HOME}/aia19_cycle24_cache_prep"
LIST_DIR="${PREP_DIR}/uri_lists"

echo "===== CYCLE-24 AIA STAGING ====="
echo "Cache root: ${CACHE_ROOT}"
echo "List dir:   ${LIST_DIR}"

mkdir -p "${CACHE_ROOT}"

AVAILABLE_KB="$(df -Pk "$(dirname "${CACHE_ROOT}")" | awk 'NR==2 {print $4}')"
AVAILABLE_GIB="$(python3 - <<PY
print(${AVAILABLE_KB}/1024/1024)
PY
)"
echo "Available GiB before staging: ${AVAILABLE_GIB}"

# 19A2B measured ~360.3 GiB payload and recommended >=432.4 GiB free.
python3 - <<PY
avail=float("${AVAILABLE_GIB}")
need=432.3897142700851
if avail < need:
    raise SystemExit(f"STOP: only {avail:.1f} GiB free; at least {need:.1f} GiB required.")
print(f"Disk gate passed: {avail:.1f} GiB free.")
PY

for list in "${LIST_DIR}"/cycle24_*_uris.txt; do
    year="$(basename "${list}" | sed -E 's/cycle24_([0-9]{4})_uris\.txt/\1/')"
    dest="${CACHE_ROOT}/${year}"
    mkdir -p "${dest}"

    echo
    echo "===== STAGING ${year} ====="
    echo "Objects: $(wc -l < "${list}")"

    # Read exact source URIs from stdin and copy them in parallel.
    gsutil -m cp -I "${dest}/" < "${list}"
done

echo
echo "===== VERIFY LOCAL FILE COUNT ====="
find "${CACHE_ROOT}" -type f -name '*.npz' | wc -l

echo "===== VERIFY DISK ====="
df -h "$(dirname "${CACHE_ROOT}")"

echo "STAGING_COMPLETE"
'''
stage_path = OUT / "stage_cycle24_aia_cache.sh"
stage_path.write_text(script)
stage_path.chmod(0o755)

print(stage_path)


## 6. Save preparation protocol

In [ ]:
summary = {
    "status": "CYCLE24_AIA_DEDUPLICATED_CACHE_PREPARED_NO_DOWNLOAD_NO_TRAINING",
    "targets": int(len(targets)),
    "positives": int(targets["label_48h_final"].sum()),
    "unique_npz_objects": int(len(objects)),
    "years": sorted(int(x) for x in objects["source_year"].unique()),
    "objects_per_year": year_counts,
    "cache_root": str(CACHE_ROOT),
    "local_path_collisions": int(len(dups)),
    "expected_payload_gib": 360.3247618917376,
    "minimum_recommended_free_gib": 432.3897142700851,
    "cycle25_used": False,
    "payload_downloaded_by_notebook": False,
    "model_trained": False,
}

(OUT / "cache_prep_summary.json").write_text(json.dumps(summary, indent=2) + "\n")
print(json.dumps(summary, indent=2))
